# 06 Full Pipeline
从 IF 信号到 Range-Doppler 图、2D CFAR 检测及结果叠加可视化。

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.core.radar_params import RadarParams
from src.core.channel_model import Target, simulate_if_signal
from src.processing.range_fft import range_fft
from src.processing.doppler_fft import range_doppler_map
from src.processing.cfar import cfar_2d
from src.visualization.spectrum_plots import plot_rd_map

params = RadarParams()
target = Target(range_m=50.0, velocity_mps=-6.0)
if_data = simulate_if_signal(params, [target])

ranges, rs = range_fft(if_data, params)
vel, rd = range_doppler_map(rs, params)
det = cfar_2d(rd)

# ── Plot 1: Interactive Range-Doppler map (Plotly) ────────────────────
fig_plotly = plot_rd_map(rd, ranges, vel)
fig_plotly.update_layout(
    title='Full Pipeline – Range-Doppler Map (50 m, −6 m/s)',
    width=750, height=450
)
fig_plotly.show()

# ── Plot 2: RD map + CFAR detections overlay (Matplotlib) ─────────────
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    rd,
    aspect='auto',
    origin='lower',
    extent=[ranges[0], ranges[-1], vel[0], vel[-1]],
    cmap='viridis',
)
plt.colorbar(im, ax=ax, label='Power (dB)')

# Overlay CFAR detection cells
r_det, v_det = np.where(det)
if r_det.size > 0:
    vel_coords = vel[r_det]
    range_coords = ranges[v_det]
    ax.scatter(range_coords, vel_coords, marker='x', color='red', s=40,
               linewidths=1.2, label='CFAR detections', zorder=5)
    ax.legend(loc='upper right')

ax.set_xlabel('Range (m)')
ax.set_ylabel('Velocity (m/s)')
ax.set_title('Range-Doppler Map with 2D CFAR Detections')
fig.tight_layout()
plt.show()

print(f'Total CFAR detections: {int(det.sum())}')
if r_det.size > 0:
    peak_r = ranges[v_det[np.argmax(rd[r_det, v_det])]]
    peak_v = vel[r_det[np.argmax(rd[r_det, v_det])]]
    print(f'Strongest detection – Range: {peak_r:.1f} m, Velocity: {peak_v:.1f} m/s')